In [ ]:
import json
import os
import random
import re
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from groq import Groq
from tqdm import tqdm

In [ ]:
load_dotenv()

INPUT_PATH = Path("../data/processed_chunks/chunks.csv")
SPLIT_PATH = Path("../data/splits/paper_split.json")
TRAIN_OUTPUT_PATH = Path("../data/synthetic_pairs/train_pairs.csv")
TEST_OUTPUT_PATH = Path("../data/synthetic_pairs/test_pairs.csv")
ALL_OUTPUT_PATH = Path("../data/synthetic_pairs/all_pairs.csv")

TRAIN_FRACTION = 0.8
RANDOM_SEED = 42
TARGET_NEW_TRAIN_PAIRS = 2000
TARGET_NEW_TEST_PAIRS = 500
MAX_ATTEMPTS = 5
REQUEST_DELAY_SECONDS = 2.1
MODEL_NAME = "llama-3.1-8b-instant"

TRAIN_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
random.seed(RANDOM_SEED)

df = pd.read_csv(INPUT_PATH)

In [ ]:
def load_or_create_split(all_papers, split_path, train_fraction=0.8, seed=42):
    all_papers = sorted(set(all_papers))

    if split_path.exists():
        split = json.loads(split_path.read_text(encoding="utf-8"))
        train_papers = sorted(set(split.get("train_papers", [])).intersection(all_papers))
        test_papers = sorted(set(split.get("test_papers", [])).intersection(all_papers))

        known = set(train_papers) | set(test_papers)
        missing = [p for p in all_papers if p not in known]

        # Assign newly added papers deterministically to train (keeps old split stable)
        train_papers = sorted(set(train_papers + missing))

    else:
        rng = random.Random(seed)
        papers = all_papers[:]
        rng.shuffle(papers)
        split_idx = int(train_fraction * len(papers))
        train_papers = sorted(papers[:split_idx])
        test_papers = sorted(papers[split_idx:])

    split = {
        "seed": seed,
        "train_fraction": train_fraction,
        "train_papers": train_papers,
        "test_papers": test_papers,
    }
    split_path.write_text(json.dumps(split, indent=2), encoding="utf-8")

    return set(train_papers), set(test_papers)


train_papers, test_papers = load_or_create_split(
    all_papers=df["paper_id"].tolist(),
    split_path=SPLIT_PATH,
    train_fraction=TRAIN_FRACTION,
    seed=RANDOM_SEED,
)

assert train_papers.isdisjoint(test_papers), "Paper leakage: train/test paper sets overlap"

train_df = df[df["paper_id"].isin(train_papers)].copy()
test_df = df[df["paper_id"].isin(test_papers)].copy()

print("papers_total:", df["paper_id"].nunique())
print("train_papers:", len(train_papers), "test_papers:", len(test_papers))
print("train_chunks:", len(train_df), "test_chunks:", len(test_df))

In [ ]:
def is_good_chunk(text):
    if not isinstance(text, str):
        return False

    t = re.sub(r"\s+", " ", text).strip()
    if len(t) < 220 or len(t) > 2600:
        return False

    # Remove obvious math/latex contamination
    banned = [r"\\usepackage", r"\\documentclass", r"\\begin\{document\}"]
    if any(re.search(p, t) for p in banned):
        return False

    # Figure/table caption-heavy chunks often generate low-quality questions
    if t.startswith("Fig.") or t.startswith("Table"):
        return False

    fig_mentions = len(re.findall(r"\b(fig\.|figure|table)\b", t, flags=re.IGNORECASE))
    if fig_mentions >= 5:
        return False

    return True


def clean_query(raw):
    if not isinstance(raw, str):
        return ""

    q = raw.replace("“", '"').replace("”", '"').strip()

    q = re.sub(
        r"^\s*(here'?s|here is|based on (the )?(given )?passage|given .*?passage)\b.*?:\s*",
        "",
        q,
        flags=re.IGNORECASE | re.DOTALL,
    )

    quoted = re.findall(r'"([^"]+\?)"', q, flags=re.DOTALL)
    if quoted:
        q = quoted[-1].strip()

    q = re.split(
        r"\b(this question is|it can be answered by|using only the information)\b",
        q,
        maxsplit=1,
        flags=re.IGNORECASE,
    )[0].strip()

    q = re.sub(r"\s+", " ", q).strip().strip(' "\'')

    m = re.search(r"(.+?\?)", q)
    if m:
        q = m.group(1).strip()

    return q


def is_valid_query(query, chunk):
    if not query:
        return False
    if len(query) < 20 or len(query) > 280:
        return False
    if not query.endswith("?"):
        return False

    lower_q = query.lower()
    banned = [
        "here's a highly specific",
        "here is a highly specific",
        "based on the passage",
        "this question is technical",
        "it can be answered",
        "information in the passage",
    ]
    if any(b in lower_q for b in banned):
        return False

    chunk_tokens = set(re.findall(r"[a-z0-9\-\+]+", str(chunk).lower()))
    query_tokens = [w for w in re.findall(r"[a-z0-9\-\+]+", lower_q) if len(w) > 2]
    if not query_tokens:
        return False

    overlap = sum(1 for w in query_tokens if w in chunk_tokens) / len(query_tokens)
    return overlap >= 0.25

In [ ]:
def load_existing(path, split_name):
    if path.exists():
        out = pd.read_csv(path)
    else:
        out = pd.DataFrame(columns=["query", "chunk", "paper_id", "chunk_id", "split"])

    if "split" not in out.columns:
        out["split"] = split_name

    out = out[["query", "chunk", "paper_id", "chunk_id", "split"]]
    out["split"] = split_name
    out = out.drop_duplicates(subset=["chunk_id"], keep="first")
    return out


api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("Missing GROQ_API_KEY in environment. Set it in .env before running.")

client = Groq(api_key=api_key)

PROMPT = """Generate exactly one biomedical research question answerable only from the passage.

Rules:
- Output only one question.
- No preface, no explanation, no quotes.
- Must end with a question mark.
- Keep it specific to passage details.

Passage:
{chunk}
"""

In [ ]:
def generate_query(chunk):
    for attempt in range(MAX_ATTEMPTS):
        try:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": "You write precise biomedical retrieval questions."},
                    {"role": "user", "content": PROMPT.format(chunk=chunk)},
                ],
                temperature=0.1,
            )
            raw = completion.choices[0].message.content or ""
            return clean_query(raw)
        except Exception as e:
            print(f"Retrying ({attempt + 1}/{MAX_ATTEMPTS}) due to: {e}")
            time.sleep(5)
    return ""


def build_pairs_for_split(source_df, output_path, split_name, target_new_pairs, seed_offset=0):
    existing = load_existing(output_path, split_name)
    processed_ids = set(existing["chunk_id"].tolist())

    eligible = source_df[source_df["text"].apply(is_good_chunk)].copy()
    eligible = eligible[~eligible["chunk_id"].isin(processed_ids)]

    if target_new_pairs <= 0 or eligible.empty:
        existing.to_csv(output_path, index=False)
        return {
            "processed": 0,
            "kept": 0,
            "skipped": 0,
            "total": len(existing),
            "eligible": len(eligible),
        }

    n = min(target_new_pairs, len(eligible))
    rows_to_process = eligible.sample(n=n, random_state=RANDOM_SEED + seed_offset)

    new_records = []
    kept = 0
    skipped = 0

    for _, row in tqdm(rows_to_process.iterrows(), total=len(rows_to_process), desc=f"Generating {split_name}"):
        chunk = row["text"]
        query = generate_query(chunk)

        if not is_valid_query(query, chunk):
            skipped += 1
            time.sleep(REQUEST_DELAY_SECONDS)
            continue

        new_records.append(
            {
                "query": query,
                "chunk": chunk,
                "paper_id": row["paper_id"],
                "chunk_id": row["chunk_id"],
                "split": split_name,
            }
        )
        kept += 1
        time.sleep(REQUEST_DELAY_SECONDS)

    if new_records:
        new_df = pd.DataFrame(new_records)
        out = pd.concat([existing, new_df], ignore_index=True)
        out = out.drop_duplicates(subset=["chunk_id"], keep="first")
    else:
        out = existing

    out.to_csv(output_path, index=False)

    return {
        "processed": len(rows_to_process),
        "kept": kept,
        "skipped": skipped,
        "total": len(out),
        "eligible": len(eligible),
    }


train_stats = build_pairs_for_split(
    source_df=train_df,
    output_path=TRAIN_OUTPUT_PATH,
    split_name="train",
    target_new_pairs=TARGET_NEW_TRAIN_PAIRS,
    seed_offset=0,
)

# Keep train untouched while building test by setting TARGET_NEW_TRAIN_PAIRS = 0
# and TARGET_NEW_TEST_PAIRS > 0 in the config cell above.
test_stats = build_pairs_for_split(
    source_df=test_df,
    output_path=TEST_OUTPUT_PATH,
    split_name="test",
    target_new_pairs=TARGET_NEW_TEST_PAIRS,
    seed_offset=1000,
)

print("train_stats:", train_stats)
print("test_stats:", test_stats)

In [ ]:
train_out = load_existing(TRAIN_OUTPUT_PATH, "train")
test_out = load_existing(TEST_OUTPUT_PATH, "test")

train_chunk_ids = set(train_out["chunk_id"].tolist())
test_chunk_ids = set(test_out["chunk_id"].tolist())
chunk_overlap = len(train_chunk_ids.intersection(test_chunk_ids))

train_paper_ids = set(train_out["paper_id"].tolist())
test_paper_ids = set(test_out["paper_id"].tolist())
paper_overlap = len(train_paper_ids.intersection(test_paper_ids))

combined = pd.concat([train_out, test_out], ignore_index=True)
combined = combined.drop_duplicates(subset=["chunk_id"], keep="first")
combined.to_csv(ALL_OUTPUT_PATH, index=False)

for name, out in [("train", train_out), ("test", test_out)]:
    q = out["query"].fillna("")
    print(f"\n{name}_rows:", len(out))
    print(f"{name}_unique_chunk_id:", out["chunk_id"].nunique())
    print(f"{name}_ends_with_question_mark:", int(q.str.endswith("?").sum()), "/", len(q))
    print(
        f"{name}_wrapper_text_count:",
        int(q.str.contains(r"here's|here is|based on the passage|this question is|it can be answered", case=False, regex=True).sum()),
    )

print("\nchunk_id_overlap_train_test:", chunk_overlap)
print("paper_id_overlap_train_test:", paper_overlap)
print("all_pairs_rows:", len(combined))

In [ ]:
# Legacy QC cell removed; use the train/test QC in the previous cell.
